In [ ]:
import pandas as pd
import numpy as np
import os
import pydicom
from tqdm import tqdm
from skimage.io import imsave
import os

In [ ]:
data_path = './manifest-1700828289467'
boxes_path = 'Annotation_Boxes.xlsx'
mapping_path = 'Breast-Cancer-MRI-filepath_filename-mapping.xlsx'
target_png_dir = 'CovertedDicomToPng'
clinical_Features = 'Clinical_and_Other_Features.xlsx'
if not os.path.exists(target_png_dir):
   os.makedirs(target_png_dir)

In [ ]:
# only consider fat-satured "pre" exams
mapping_df = pd.read_excel(mapping_path)
boxes_df = pd.read_excel(boxes_path)
clinical_Features_df = pd.read_excel(clinical_Features)
mapping_df = mapping_df[mapping_df['original_path_and_filename'].str.contains('pre')]

# remove entries from patients that we are not including (we only include patients 1 to 10)
crossref_pattern = '|'.join(["DICOM_Images/Breast_MRI_{:03d}".format(s) for s in list(range(801, 923))])
mapping_df = mapping_df[mapping_df['original_path_and_filename'].str.contains(crossref_pattern)]

In [ ]:
def save_dcm_slice(dcm_fname, label, vol_idx):
    # create a path to save the slice .png file in, according to the original DICOM filename and target label
    png_path = dcm_fname.split('/')[-1].replace('.dcm', '-{}.png'.format(vol_idx))
    label_dir = 'class_'+str(label)
    png_path = os.path.join(target_png_dir, label_dir, png_path)

    if not os.path.exists(os.path.join(target_png_dir, label_dir)):
        os.makedirs(os.path.join(target_png_dir, label_dir))

    if not os.path.exists(png_path):
        # only make the png image if it doesn't already exist (if you're running this after the first time)

        # load DICOM file with pydicom library
        try:
            dcm = pydicom.dcmread(dcm_fname)
        except FileNotFoundError:
            # fix possible errors in filename from list
            dcm_fname_split = dcm_fname.split('/')
            dcm_fname_end = dcm_fname_split[-1]
            assert dcm_fname_end.split('-')[1][0] == '0'

            dcm_fname_end_split = dcm_fname_end.split('-')
            dcm_fname_end = '-'.join([dcm_fname_end_split[0], dcm_fname_end_split[1][1:]])

            dcm_fname_split[-1] = dcm_fname_end
            dcm_fname = '/'.join(dcm_fname_split)
            dcm = pydicom.dcmread(dcm_fname)


        # convert DICOM into numerical numpy array of pixel intensity values
        img = dcm.pixel_array

        # convert uint16 datatype to float, scaled properly for uint8
        img = img.astype(float) * 255. / img.max()
        # convert from float -> uint8
        img = img.astype(np.uint8)
        # invert image if necessary, according to DICOM metadata
        img_type = dcm.PhotometricInterpretation
        if img_type == "MONOCHROME1":
            img = np.invert(img)

        # save final .png
        imsave(png_path, img)
    return png_path

In [ ]:
class_df = pd.DataFrame(columns=['image_title','class_label'])

In [ ]:
# initialize iteration index of each patient volume
vol_idx = -1

for row_idx, row in tqdm(mapping_df.iterrows()):
    # indices start at 1 here
    new_vol_idx = int((row['original_path_and_filename'].split('/')[1]).split('_')[-1])
    slice_idx = int(((row['original_path_and_filename'].split('/')[-1]).split('_')[-1]).replace('.dcm', ''))

    # new volume: get tumor bounding box
    if new_vol_idx != vol_idx:
        box_row = boxes_df.iloc[[new_vol_idx-1]]
        start_slice = int(box_row['Start Slice'])
        end_slice = int(box_row['End Slice'])
        assert end_slice >= start_slice
    vol_idx = new_vol_idx

    

    # get DICOM filename
    dcm_fname = str(row['classic_path'])
    dcm_fname = os.path.join(data_path, dcm_fname)

    # determine slice label:
    # (1) if within 3D box, save as positive
    if slice_idx >= start_slice and slice_idx < end_slice:        
        if not pd.isna(clinical_Features_df.iloc[vol_idx-1]['Staging(Nodes)#(Nx replaced by -1)[N]']):
            path = save_dcm_slice(dcm_fname, int(clinical_Features_df.iloc[vol_idx-1]['Staging(Nodes)#(Nx replaced by -1)[N]']), vol_idx)
            new_Class_Row = [os.path.basename(path) , int(clinical_Features_df.iloc[vol_idx-1]['Staging(Nodes)#(Nx replaced by -1)[N]'])]
            #new_Class_Row = ['1-'+str(slice_idx)+'-'+str(vol_idx)+'.png' , int(clinical_Features_df.iloc[vol_idx-1]['Tumor Grade Tubule'])]
            class_df = pd.concat([class_df, pd.DataFrame([new_Class_Row],columns=['image_title','class_label'])], ignore_index=True)

In [ ]:
class_df.to_csv('classPerImage.csv')

In [ ]:
class_df